# 08 · Evaluate the 3D segmenter: tune on val, then test once

1. Clone the repo and install the package (+ MONAI).
2. **Val:** run the trained SegResNet on the 188 val patients and pick the small-ET rule: predicted enhancing tumor under N voxels (mm³) is dropped, N chosen for the best val ET Dice.
3. **Test, once:** run the 188 test patients with that N. Test results are reported raw and with the rule.

**Kaggle setup:** GPU T4 x2, Internet on, inputs `dschettler8845/brats-2021-task1` and the output of the `07_segmentation` notebook (it holds `segmenter_segresnet.pt`). Run with *Save Version → Save & Run All*, then download everything from *Output*.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sanjardevx/neurolens.git"
BRANCH = "develop"
CONFIG = "ml/configs/seg_segresnet.yaml"
REPO_DIR = Path(os.environ.get("NEUROLENS_REPO_DIR", "/tmp/neurolens"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXTRA_ARGS = os.environ.get("NEUROLENS_ARGS", "").split()  # only for local smoke tests (fake data, checkpoint)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# --no-deps: keep Kaggle's own torch and numpy; MONAI and nibabel need nothing else
missing = [p for p in ["neurolens_ml", "monai", "nibabel"] if importlib.util.find_spec(p) is None]
if "neurolens_ml" in missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR / "ml")], check=True)
if pkgs := [p for p in missing if p != "neurolens_ml"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pkgs], check=True)
print("repo commit:", commit)

In [ ]:
# Val first: it writes the ET size limit. Test refuses to run without it, and runs once.
for split in ["val", "test"]:
    cmd = [sys.executable, "-m", "neurolens_ml.evaluate_seg", "--config", str(REPO_DIR / CONFIG), "--split", split,
           "--out", str(OUT_DIR), *EXTRA_ARGS]
    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, f"{split} evaluation failed, see the log above"

In [ ]:
import pandas as pd
import yaml

name = yaml.safe_load((REPO_DIR / CONFIG).read_text())["name"]
metrics = json.loads((OUT_DIR / f"metrics_seg_test_{name}.json").read_text())
metrics["repo_commit"] = commit
(OUT_DIR / f"metrics_seg_test_{name}.json").write_text(json.dumps(metrics, indent=2))
print("ET size limit (from val):", metrics["min_et_voxels"], "voxels")
pd.DataFrame({"test, raw": metrics["test_raw"], "test, with the rule": metrics["test"]}).T